# 4.7 Demo - Continuous Integration and Continuous Deployment with DABs
**Lessons:** CI/CD Project Overview with DABs · CI/CD Best Practices for Data Engineering · Next Steps: Automated Deployment with GitHub Actions

Project: `bundles/03_cicd_dab`. A Python package (`orders_lib`) with pure transformation functions, **unit tests** that run on local Spark, a **wheel artifact** built by the bundle, and a serverless job. Three targets: dev → stage → prod.

![GitHub Actions pipeline for bundles/03_cicd_dab: ci job, deploy-stage, approval, deploy-prod](../images/4.7-cicd-pipeline.svg)

**How to read it:** each row is one trigger of `ci/github-actions/bundle-ci-cd.yml`. Every trigger runs the green `ci` job (sections B and D
below); a PR stops there, a push to main continues to stage (section E), a release waits for reviewers before prod.
Bottom: the secrets are passed by name only, and `deploy` builds the wheel (section C) that the job installs from `../dist/*.whl`.

This notebook runs each pipeline step locally, in order, as a CI runner would.

In [ ]:
import subprocess, sys
from adelab import bundles
from adelab.spark import PROJECT_ROOT

BUNDLE = PROJECT_ROOT / "04-automated-deployment-dabs" / "bundles" / "03_cicd_dab"
status = bundles.workspace_status()
bundles.tree(BUNDLE)

def step(name, cmd):
    print(f"\n▶ {name}: {' '.join(cmd)}")
    p = subprocess.run(cmd, cwd=BUNDLE, capture_output=True, text=True)
    print((p.stdout + p.stderr)[-2500:])
    if p.returncode != 0:
        raise SystemExit(f"step '{name}' failed. The pipeline stops here, like a red CI build")

## A. Testable code: logic in functions, not in notebooks

In [ ]:
print((BUNDLE / "src" / "orders_lib" / "transforms.py").read_text())
print((BUNDLE / "tests" / "test_transforms.py").read_text())

## B. CI step 1: unit tests
Fast, local, no workspace needed. Run them on every commit.

In [ ]:
step("unit tests", [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"])

## C. CI step 2: build the artifact
`databricks bundle deploy` runs the `artifacts.orders_lib.build` command from `databricks.yml` and uploads the wheel. Run the same command here:

In [ ]:
step("build wheel", [sys.executable, "-m", "pip", "wheel", "--no-deps", "-q", "-w", "dist", "."])
print([p.name for p in (BUNDLE / "dist").glob("*.whl")])

## D. CI step 3: validate the bundle for every target

In [ ]:
assert not bundles.validate_schema(BUNDLE)
for t in ("dev", "stage", "prod"):
    cfg = bundles.preview(BUNDLE, t, show=False)
    task = cfg["resources"]["jobs"]["orders_cicd_job"]["tasks"][0]
    print(f"{t:<6} job={cfg['resources']['jobs']['orders_cicd_job']['name']:<32} params={task['python_wheel_task']['parameters']}")

if bundles.require_workspace(status):
    for t in ("dev", "stage", "prod"):
        bundles.cli("bundle", "validate", "-t", t, cwd=BUNDLE)

## E. CD: deploy to stage and run an integration test

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "deploy", "-t", "stage", cwd=BUNDLE)
    bundles.cli("bundle", "run", "-t", "stage", "orders_cicd_job", cwd=BUNDLE)

## F. The same pipeline in GitHub Actions
`ci/github-actions/bundle-ci-cd.yml` automates steps B-E. To use it, copy it to `.github/workflows/` of the repository and configure these **secrets** (names only, never commit values): `DATABRICKS_HOST`, `DATABRICKS_CLIENT_ID`, `DATABRICKS_CLIENT_SECRET` (a service principal with OAuth M2M). Protect the `prod` GitHub environment with required reviewers.

In [ ]:
print((BUNDLE / "ci" / "github-actions" / "bundle-ci-cd.yml").read_text())

## G. CI/CD best practices for data engineering
- **Everything as code**: jobs, pipelines, permissions, schemas, and dashboards in the bundle. No manual changes in stage or prod.
- **Test pyramid**: many fast unit tests (pure functions on small DataFrames), fewer integration tests (run the job on stage with representative data), data quality expectations in production.
- **Isolated environments**: separate catalogs or schemas per target. Ideally separate workspaces for prod.
- **Identities**: stage and prod run as **service principals** (`run_as`). Humans get `CAN_VIEW` / `CAN_MANAGE_RUN`, not ownership.
- **Promotion, not rebuilds**: deploy the same Git commit to stage and then prod. Differences are only variables.
- **Secrets** in secret scopes or the CI secret store. Reference them by name only.

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "destroy", "-t", "stage", "--auto-approve", cwd=BUNDLE)